# Assignment 8 — Movie Recommendation Systems

**Juan Luna Santibanez**  
**Miami Dade College | CAI2100C — Machine Learning Foundations**  
**Professor Beth Muturi**

## My two assigned tasks

1. **Task 1 — Content-Based Filtering:** I use IMDb movie metadata, TF-IDF and cosine similarity to recommend related films.
2. **Task 2 — Collaborative Filtering:** I compare rating patterns from IMDb's available demographic cohorts, evaluate held-out group averages and describe the limitations of aggregate ratings.

I use the **two complete original IMDb Extensive CSV files**, stored in the project's `data/` folder. As in my Assignment 7, the notebook checks for local CSV files and downloads them from my public GitHub repository only when needed. The accompanying ZIP includes the original CSVs for offline data access.

**Data scope:** the original IMDb movies and ratings tables (85,855 records each). No fixed 3,000-title sample or popularity cutoff is applied. Any titles omitted due to missing required identifiers are reported explicitly.

[IMDb Extensive dataset](https://www.kaggle.com/datasets/stefanoleone992/imdb-extensive-dataset) · [Assignment 8 GitHub repository](https://github.com/SupaHumano/CAI2100C-Movie-Recommendation-Systems)


## 1. Load and inspect IMDb Extensive

I use the same straightforward data-loading approach as Assignment 7. My two unmodified source files are `IMDb movies.csv` and `IMDb ratings.csv` in `data/`. The loader searches local project directories first. For Colab, where I may open the notebook without cloning the project, it retrieves missing files from my GitHub repository. The supplied ZIP already includes both full CSVs.


In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

np.random.seed(42)
print('Python:', platform.python_version(), '| Pandas:', pd.__version__)

def recommend_by_content(frame, matrix, selected_title, k=10):
    found = np.flatnonzero(frame.title.astype(str).str.casefold().eq(selected_title.casefold()))
    if not len(found):
        raise ValueError('Title not found: '+selected_title)
    target = int(found[0])
    similarity = linear_kernel(matrix[target],matrix).ravel()
    indices = np.argsort(-similarity,kind='stable')
    indices = indices[indices!=target][:k]
    result = frame.iloc[indices][['title']].copy()
    result['cosine_similarity']=similarity[indices]
    return result.reset_index(drop=True)


### Original IMDb Extensive files

I use the complete IMDb Extensive movie and rating tables, rather than a smaller popularity-based sample. The source files each contain **85,855 rows**. The notebook reports their actual row counts and processes the available valid film identifiers, explaining any exclusions required for the algorithms.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

# Assignment 7 pattern: local CSVs first; download missing files from my GitHub repository.
DATA_BASE = "https://raw.githubusercontent.com/SupaHumano/CAI2100C-Movie-Recommendation-Systems/main/data/"
ORIGINAL_FILE_MIN_BYTES = {"IMDb movies.csv": 45_000_000, "IMDb ratings.csv": 16_000_000}

def locate_csv(filename):
    cwd = Path.cwd()
    candidates = [
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        cwd / filename,
        cwd / "notebooks" / "data" / filename,
        Path("/content/data") / filename,
        Path("/content") / filename,
    ]
    if Path("/kaggle/input").is_dir():
        candidates += list(Path("/kaggle/input").glob("**/" + filename))
    source = next((p for p in candidates if p.is_file() and p.stat().st_size >= ORIGINAL_FILE_MIN_BYTES[filename]), None)
    if source is None:
        source = cwd / "data" / filename
        source.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(DATA_BASE + filename.replace(" ", "%20"), source)
    return source

movies_path = locate_csv("IMDb movies.csv")
ratings_path = locate_csv("IMDb ratings.csv")
imdb_movies_raw = pd.read_csv(movies_path, low_memory=False)
imdb_ratings_raw = pd.read_csv(ratings_path, low_memory=False)
assert len(imdb_movies_raw) == 85855 and len(imdb_ratings_raw) == 85855, "Expected complete original IMDb Extensive CSVs"
assert imdb_movies_raw.imdb_title_id.is_unique
assert imdb_ratings_raw.imdb_title_id.is_unique
print("IMDb movies:", len(imdb_movies_raw), "| source:", movies_path)
print("IMDb rating records:", len(imdb_ratings_raw), "| source:", ratings_path)
display(imdb_movies_raw[["title", "genre", "votes", "avg_vote"]].head())


## Task 1 — Content-Based Filtering

I represent each film using **genre, director, actors and description**. TF-IDF converts those attributes to numeric features, and **cosine similarity** ranks related titles. I compute similarity only for the selected example movie rather than constructing a large all-movies-by-all-movies matrix.

I train the vectorizer on the **entire IMDb movie table** (apart from any records without a usable film identifier), without reducing it to 3,000 titles. Content similarity alone does not mean that a particular viewer will enjoy the movie.


In [ ]:
COHORTS = {
 'Women 18–29': ('females_18age_avg_vote', 'females_18age_votes'),
 'Women 30–44': ('females_30age_avg_vote', 'females_30age_votes'),
 'Women 45+': ('females_45age_avg_vote', 'females_45age_votes'),
 'Men 18–29': ('males_18age_avg_vote', 'males_18age_votes'),
 'Men 30–44': ('males_30age_avg_vote', 'males_30age_votes'),
 'Men 45+': ('males_45age_avg_vote', 'males_45age_votes')}
cohort_cols = [c for pair in COHORTS.values() for c in pair]
assert {'imdb_title_id','title','genre','description','director','actors','votes','avg_vote'}.issubset(imdb_movies_raw.columns)
assert {'imdb_title_id', *cohort_cols}.issubset(imdb_ratings_raw.columns)

# Keep the complete original catalogue: no popularity cutoff, no head(3000).
imdb_movies = imdb_movies_raw.copy()
imdb_movies['votes'] = pd.to_numeric(imdb_movies['votes'], errors='coerce').fillna(0)
imdb_movies['avg_vote'] = pd.to_numeric(imdb_movies['avg_vote'], errors='coerce')
imdb_movies = imdb_movies[imdb_movies.imdb_title_id.notna()].drop_duplicates('imdb_title_id').copy()
for col in ['title','genre','description','director','actors']:
    imdb_movies[col] = imdb_movies[col].fillna('').astype(str)
imdb_ratings_unique = imdb_ratings_raw[['imdb_title_id'] + cohort_cols].drop_duplicates('imdb_title_id')
imdb_movies = imdb_movies.merge(imdb_ratings_unique, on='imdb_title_id', how='left', validate='one_to_one').reset_index(drop=True)
assert len(imdb_movies) >= 85000
print('Original film records:',len(imdb_movies_raw),'| usable catalogue records:',len(imdb_movies))
display(imdb_movies[['title','genre','votes','avg_vote']].head())


In [ ]:
imdb_text = (imdb_movies.genre.str.replace(',', ' ', regex=False) + ' ' +
             imdb_movies.director.str.replace(',', ' ', regex=False) + ' ' +
             imdb_movies.actors.str.replace(',', ' ', regex=False) + ' ' + imdb_movies.description)
imdb_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=6000, dtype=np.float32)
imdb_features = imdb_vectorizer.fit_transform(imdb_text)
# Choose a well-rated, high-vote demonstration movie without excluding other movies from the model.
imdb_title = str(imdb_movies.loc[imdb_movies.votes.idxmax(),'title'])
imdb_content = recommend_by_content(imdb_movies,imdb_features,imdb_title,k=10)
print('IMDb film catalogue for content filtering:',len(imdb_movies))
print('Selected example title:',imdb_title)
display(imdb_content.round(4))
assert len(imdb_content) == 10
fig,ax = plt.subplots(figsize=(9,4.5)); table=imdb_content.iloc[::-1]
ax.barh(table.title.str.slice(0,38), table.cosine_similarity)
ax.set(title='IMDb Extensive — Content-Based recommendations',xlabel='TF-IDF cosine similarity')
fig.tight_layout();plt.show()


## Task 2 — Collaborative Filtering approximation

The IMDb Extensive ratings file supplies **aggregated gender/age cohort averages**, not individual `userId–movieId–rating` events. I therefore compare six demographic groups, rather than claiming personalized user-based collaborative filtering or SVD.

I use **all valid available group/movie observations across the full catalogue**, with a minimum of 30 votes per cohort rating. I calculate Pearson correlations between cohorts, then withhold approximately 15% of eligible group averages to evaluate the model using MAE, RMSE and a training-group-mean baseline. The scores are errors on the 1–10 IMDb rating scale.


In [ ]:
def make_cohort_matrix(frame,min_votes=30):
    rows=[]
    for mean_col,votes_col in COHORTS.values():
        mean=pd.to_numeric(frame[mean_col],errors='coerce')
        nvotes=pd.to_numeric(frame[votes_col],errors='coerce').fillna(0)
        rows.append(mean.where(mean.between(1,10)&nvotes.ge(min_votes)).to_numpy(float))
    return pd.DataFrame(rows,index=list(COHORTS),columns=frame.imdb_title_id.tolist())

def group_similarity(groups,min_overlap=15):
    x=groups.to_numpy(float);n=x.shape[0];sim=np.eye(n)
    for i in range(n):
        for j in range(i+1,n):
            valid=np.isfinite(x[i])&np.isfinite(x[j])
            if valid.sum()>=min_overlap and np.std(x[i,valid])>1e-12 and np.std(x[j,valid])>1e-12:
                s=np.corrcoef(x[i,valid],x[j,valid])[0,1]
                sim[i,j]=sim[j,i]=float(np.clip(s,-1,1)) if np.isfinite(s) else 0.0
            else:
                sim[i,j]=sim[j,i]=0.0
    return pd.DataFrame(sim,index=groups.index,columns=groups.index)

def predict_group_average(groups,similarity,row,col,group_av=None,data=None):
    x=groups.to_numpy(float) if data is None else data
    group_av=np.nanmean(x,axis=1) if group_av is None else group_av
    base=float(group_av[row]) if np.isfinite(group_av[row]) else float(np.nanmean(x))
    w=similarity.iloc[row].to_numpy(float).copy();w[row]=0
    good=(w>0)&np.isfinite(x[:,col])&np.isfinite(group_av)
    if not good.any():return float(np.clip(base,1,10))
    result=base+np.dot(w[good],x[good,col]-group_av[good])/w[good].sum()
    return float(np.clip(result,1,10))

def evaluate_groups(groups,seed=42,fraction=0.15):
    x=groups.to_numpy(float).copy()
    pairs=np.argwhere(np.isfinite(x))
    if len(pairs)<60:raise ValueError('Need at least 60 observed demographic averages for validation')
    rng=np.random.default_rng(seed);rng.shuffle(pairs)
    row_counts=np.isfinite(x).sum(axis=1)
    col_counts=np.isfinite(x).sum(axis=0)
    withheld=[];max_holdout=max(1,round(len(pairs)*fraction))
    for row,col in pairs:
        if len(withheld)>=max_holdout:break
        if row_counts[row]<=15 or col_counts[col]<=2:continue
        withheld.append((int(row),int(col),float(x[row,col])))
        x[row,col]=np.nan
        row_counts[row]-=1;col_counts[col]-=1
    if not withheld:raise ValueError('Unable to select any evaluable holdout')
    train=pd.DataFrame(x,index=groups.index,columns=groups.columns)
    sims=group_similarity(train)
    av=np.nanmean(x,axis=1)
    y_true=np.array([value for _,_,value in withheld])
    y_pred=np.array([predict_group_average(train,sims,row,col,av,x) for row,col,_ in withheld])
    baseline=np.array([av[row] for row,_,_ in withheld])
    diff=y_pred-y_true
    return {'Held-out group means':len(y_true),'MAE':float(np.mean(abs(diff))),
            'RMSE':float(np.sqrt(np.mean(diff**2))),
            'Training-group-mean baseline MAE':float(np.mean(abs(baseline-y_true)))},train

imdb_groups=make_cohort_matrix(imdb_movies)
imdb_sim=group_similarity(imdb_groups)
print('IMDb matrix dimensions (groups × full movie catalogue):',imdb_groups.shape)
display(imdb_sim.round(3))
imdb_scores,imdb_train_groups=evaluate_groups(imdb_groups)
display(pd.DataFrame([imdb_scores]).T.rename(columns={0:'Value'}))
assert np.isfinite(imdb_scores['MAE'])
fig,ax=plt.subplots(figsize=(7,5))
heat=ax.imshow(imdb_sim.to_numpy(),vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(6),imdb_sim.columns,rotation=45,ha='right')
ax.set_yticks(range(6),imdb_sim.index)
fig.colorbar(heat,ax=ax,label='Pearson correlation');fig.tight_layout();plt.show()


In [ ]:
# A group-level recommendation example using all candidate films in the complete IMDb collection.
chosen_group='Women 18–29';row=list(imdb_groups.index).index(chosen_group)
available=np.flatnonzero(imdb_groups.iloc[row].isna().to_numpy() & (imdb_groups.notna().sum(axis=0).to_numpy()>=2))
reason='Estimated missing group averages'
if not len(available):
    available=np.flatnonzero(imdb_groups.notna().sum(axis=0).to_numpy()>=3)
    reason='Illustrative estimates for films with observed group ratings'
if not len(available):raise ValueError('Not enough IMDb group observations for ranking')
mat=imdb_groups.to_numpy(float)
means=np.nanmean(mat,axis=1)
est=np.array([predict_group_average(imdb_groups,imdb_sim,row,int(j),means,mat) for j in available])
order=np.argsort(-est)[:10]
keys=imdb_groups.columns[available[order]]
imdb_group_top=imdb_movies.set_index('imdb_title_id').loc[keys,['title','genre','avg_vote']].copy()
imdb_group_top['estimated_group_average']=est[order]
print('IMDb group:',chosen_group,'|',reason)
display(imdb_group_top.reset_index())


## My results and interpretation

I compare the two **required IMDb methods**, not different rating scales or unrelated datasets. Content-Based Filtering yields a ranked list of similar movies, whereas the cohort-correlation approach produces estimated **group-level averages**. These methods answer different questions, so I do not compare their cosine similarity values with MAE/RMSE as if they were identical measurements.


In [ ]:
summary = pd.DataFrame([
 {'Task':'1 — Content-Based', 'Data':'Movie descriptions and metadata', 'Result':f'{len(imdb_content)} recommended films', 'Measurement':'TF-IDF cosine similarity'},
 {'Task':'2 — Cohort rating patterns', 'Data':'Six demographic rating groups', 'Result':f'{imdb_scores["Held-out group means"]} held-out group means', 'Measurement':f'MAE {imdb_scores["MAE"]:.4f}; RMSE {imdb_scores["RMSE"]:.4f}; baseline MAE {imdb_scores["Training-group-mean baseline MAE"]:.4f}'}
])
display(summary)
print('PASS — Both assigned IMDb recommendation methods completed from the two complete IMDb source CSV files.')


## My conclusion

My **Content-Based** method ranks movies by metadata similarities using TF-IDF and cosine similarity across the full IMDb Extensive movie catalogue. My second approach studies **aggregated demographic rating patterns** and evaluates predicted cohort averages, which is different from personal, user-based collaborative filtering.

**Data scope:** I read the two complete original IMDb Extensive CSV tables of 85,855 records each. I did not discard films because of popularity or select a 3,000-title subset. My recommender uses every movie with a valid identifier, and Task 2 uses the cohort ratings that satisfy its stated minimum vote threshold.

**Methodological limitation:** cohort averages cannot determine how any particular viewer rated a film. Held-out MAE/RMSE evaluates group-level predictions, not individual preferences.

**Sources:** [Stefano Leone, IMDb Extensive](https://www.kaggle.com/datasets/stefanoleone992/imdb-extensive-dataset); Professor Beth Muturi's Assignment 8 notebook; [scikit-learn TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
